# 3. Fit two tiers and tune incident persistence on validation only
References and Isolation Forest use training data. A separate chronological normal
calibration segment maps raw scores to [0, 1]. These are interpolated ranks, not
probabilities. Statistical EWMA, Isolation Forest and their maximum are compared.
The maximum changes the false-alarm rate and is not automatically preferred.
Five cumulative feature sets are compared using the same time splits and incident policies.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation
from optical_anomaly.workflow import run_split

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
split = run_split(settings)
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    split.train_end,
    split.calibration_end,
    split.validation_end,
    split.test_end,
]
REPORT = RUN / "eda"
REPORT.mkdir(exist_ok=True)
print("Run:", RUN.resolve())

In [ ]:
if not (RUN / "model.joblib").exists():
    develop(CONFIG_PATH)
comparison = pd.read_csv(RUN / "validation_comparison.csv")
columns = [
    "feature_set",
    "feature_count",
    "score_coverage",
    "detector",
    "opening_intervals",
    "closing_intervals",
    "detected",
    "faults",
    "warning_opportunities",
    "pre_impact_recall",
    "minimum_lead_recall",
    "pre_impact_recall_all_impacting",
    "faults_without_warning_opportunity",
    "unmatched_incidents",
    "duplicate_incidents",
    "nuisance_per_1000_entity_days",
    "meets_workload_budget",
]
display(comparison[columns])
manifest = json.loads((RUN / "manifest.json").read_text())
print("Frozen experimental policy:", manifest["policy"])
print("Meets validation workload budget:", manifest["meets_validation_workload_budget"])

validation_scores = pd.read_parquet(RUN / "validation_scores.parquet")
display(
    validation_scores[["statistical", "isolation_forest", "combined"]]
    .notna()
    .mean()
    .rename("score_coverage")
)
display(pd.read_csv(RUN / "feature_set_comparison.csv"))
print("Selected feature set:", manifest["feature_set"])

Telemetry scope is explicit in `model.feature_set` (default `temperature`, meaning the full set including temperature).
Comparisons never automatically remove features or select a smaller telemetry set.
Within the configured set, validation chooses a detector and incident persistence
policy using workload feasibility, minimum-lead recall and nuisance workload. The budget affects
incident-policy selection, not feature retention. A failing policy is flagged.
Notebook 06 explains each fitted forest without modifying it. Validation remains
development evidence; use a new output folder for changed settings or code.